In [ ]:
# !pip3 install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


# Generate stories TinyStories-33M

In [2]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import Markdown, display


# Enable MPS fallback for operations not supported on MPS
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Check compute device and capabilities
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

# Determine device to use
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA device: {device}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using MPS device: {device}")
else:
    device = torch.device("cpu")
    print(f"Using CPU device: {device}")

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained('roneneldan/TinyStories-33M')
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")

# Move model to appropriate device
model = model.to(device)
print(f"Model moved to device: {next(model.parameters()).device}")

# Set your prompt
prompt = "There once was a man who was a great hunter. He was known for his skill and "
print(f"\nPrompt: {prompt}")

# Process on selected device
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
attention_mask = torch.ones_like(input_ids).to(device)

# Generate text
output = model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_length=200,
    num_beams=1
)
story = tokenizer.decode(output[0], skip_special_tokens=True)

print("\nGenerated story:")
display(Markdown(story))

/Users/santiago/Documents/Personal/Tesis/Codigo/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.9.0.dev20250714
CUDA available: False
MPS available: True
Using MPS device: mps


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Model moved to device: mps:0

Prompt: There once was a man who was a great hunter. He was known for his skill and 

Generated story:


There once was a man who was a great hunter. He was known for his skill and witched with ease. One day, he was walking through the forest when he heard a noise. He stopped and listened carefully. He heard a voice calling out to him. He followed the voice and soon he came to a clearing. In the clearing was a little girl. She was only three years old.

The hunter asked her what she was doing. She said she was trying to find her way home. The hunter smiled and said he would help her. He took her hand and they started walking together.

The hunter and the little girl walked for a long time. The little girl was getting tired, but the hunter kept her safe. Finally, they reached the little girl's house. The hunter said goodbye and the little girl thanked him.

The hunter was happy he could help the little girl. He smiled and waved goodbye. He was glad he could make a new friend

In [3]:
def generate_story(model_name, prompt):
    """
    Generate a story using the specified model and prompt.
    
    Args:
        model_name (str): The name of the model to use for generation
        prompt (str): The prompt to generate the story from
    
    Returns:
        str: The generated story
    """
    # Load model and tokenizer
    model = AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
    
    # Move model to appropriate device
    model = model.to(device)
    print(f"Model moved to device: {next(model.parameters()).device}")
    
    print(f"\nPrompt: {prompt}")
    
    # Process on selected device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    attention_mask = torch.ones_like(input_ids).to(device)
    
    # Generate text
    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=200,
        num_beams=1
    )
    story = tokenizer.decode(output[0], skip_special_tokens=True)
    
    print("\nGenerated story:")
    display(Markdown(story))
    
    return story

model_name='roneneldan/TinyStories-33M'
prompt="There once was a robot"

# Example usage
story = generate_story(model_name, prompt)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Model moved to device: mps:0

Prompt: There once was a robot

Generated story:


There once was a robot who lived in a big city. He was very happy and loved to play with his friends. One day, he saw a little girl who was lost and crying. He went to her and asked her what was wrong. She said she was lost and couldn't find her mommy. The robot knew he had to help her, so he took her to his mommy who was very happy to see her.

The robot was very kind and gentle with the little girl. He told her that everything was going to be okay and that he would help her find her mommy. The little girl was so happy and thanked the robot for being so helpful.

After a while, the robot had to go back to his home in the city. He said goodbye to the little girl and her mommy and went back to their big city. The robot felt happy that he could help someone and he knew that he had done a good thing.


# Tiny stories instruct

In [4]:
def chat_with_model(model_name):
    """
    Interactive chat function that takes user inputs and generates responses using the specified model.
    
    Args:
        model_name (str): The name of the model to use for generation (e.g., 'roneneldan/TinyStories-Instruct-8M')
    """
    # Load model and tokenizer
    model = AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
    
    # Move model to appropriate device
    model = model.to(device)
    print(f"Model moved to device: {next(model.parameters()).device}")
    print(f"Chat started with model: {model_name}")
    print("Type 'quit' to exit the chat\n")
    
    while True:
        # Get user input
        user_input = input("You: ")
        
        # Check if user wants to quit
        if user_input.lower() in ['quit', 'exit', 'q']:
            break
        
        # Process on selected device
        input_ids = tokenizer.encode(user_input, return_tensors="pt").to(device)
        attention_mask = torch.ones_like(input_ids).to(device)
        
        # Generate text
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=200,
            num_beams=1
        )
        response = tokenizer.decode(output[0], skip_special_tokens=False)
        
        print("\nModel:")
        display(Markdown(response))
        print()

# Chat with roneneldan/TinyStories-Instruct-8M

In [5]:
chat_with_model('roneneldan/TinyStories-Instruct-8M')

Model moved to device: mps:0
Chat started with model: roneneldan/TinyStories-Instruct-8M
Type 'quit' to exit the chat



In [6]:
chat_with_model('roneneldan/TinyStories-Instruct-2Layers-33M')

Model moved to device: mps:0
Chat started with model: roneneldan/TinyStories-Instruct-2Layers-33M
Type 'quit' to exit the chat



# Weighted Word Generation

Let's implement a custom logits processor to weight specific words in the probability distribution during generation.


In [7]:
from transformers.generation import LogitsProcessor, LogitsProcessorList
import torch

class ProbabilityWeightingLogitsProcessor(LogitsProcessor):
    """Custom logits processor that properly modifies the probability of specific tokens."""
    
    def __init__(self, tokenizer, words_to_weight, weight_factor, verbose):
        self.tokenizer = tokenizer
        self.word_token_ids = {}  # Dictionary to store word -> token_ids mapping
        self.verbose = verbose
        
        if self.verbose:
            print("\nPrompt token mapping:")
            for i, token_id in enumerate(input_ids[0].tolist()):
                token = tokenizer.decode([token_id])
                print(f"Position {i}: Token ID {token_id} → '{token}'")
            
            print("\nWeighted words token mapping:")
        
        # For each word, get token IDs both with and without space prefix
        for word in words_to_weight:
            # Get token IDs for the word without space prefix
            word_ids = tokenizer.encode(word, add_special_tokens=False)
            
            # Get token IDs for the word with space prefix
            word_with_space_ids = tokenizer.encode(" " + word, add_special_tokens=False)
            
            # Store both versions
            self.word_token_ids[word] = {
                'without_space': word_ids,
                'with_space': word_with_space_ids
            }
            
            if self.verbose:
                print(f"Word '{word}':")
                print(f"  Without space prefix: {word_ids} → '{tokenizer.decode(word_ids)}'")
                print(f"  With space prefix: {word_with_space_ids} → '{tokenizer.decode(word_with_space_ids)}'")
        
        self.weight_factor = weight_factor
        if self.verbose:
            print(f"Applying weight factor: {weight_factor}")
            print()
    
    def __call__(self, input_ids, scores):
        # Get the current sequence
        if self.verbose:
            current_text = tokenizer.decode(input_ids[0])
            print(f"Current sequence: '{current_text}'")
            
            # Get top tokens before weighting
            probs = torch.nn.functional.softmax(scores, dim=-1)
            top_probs, top_indices = torch.topk(probs[0], 5)
            print("Top 5 next tokens before weighting:")
            for i, (token_id, prob) in enumerate(zip(top_indices.tolist(), top_probs.tolist())):
                token = tokenizer.decode([token_id])
                print(f"  {i+1}. Token ID {token_id} → '{token}' (prob: {prob:.6f})")
        else:
            probs = torch.nn.functional.softmax(scores, dim=-1)
        
        # Track if any weights were applied
        weights_applied = False
        
        # Apply weighting to probabilities for all token variations
        for word, token_ids_dict in self.word_token_ids.items():
            # Check both with and without space versions
            for prefix_type, token_ids in token_ids_dict.items():
                for token_id in token_ids:
                    if token_id < scores.shape[-1]:  # Make sure token_id is within vocabulary
                        # Convert to probability space
                        original_prob = probs[0, token_id].item()
                        
                        # Apply weight to probability
                        weighted_prob = original_prob * self.weight_factor
                        
                        # Ensure probability is valid (between 0 and 1)
                        weighted_prob = max(0, min(1, weighted_prob))
                        
                        # Update the probability
                        probs[0, token_id] = weighted_prob
                        
                        # Check if this weight actually made a difference
                        if abs(weighted_prob - original_prob) > 1e-6:
                            weights_applied = True
                        
                        if self.verbose:
                            token = tokenizer.decode([token_id])
                            print(f"Weighted: Token ID {token_id} → '{token}' (prob: {original_prob:.6f} → {weighted_prob:.6f})")
        
        # Renormalize probabilities to sum to 1
        probs = probs / probs.sum(dim=-1, keepdim=True)
        
        # Convert back to logits
        # We use a numerically stable approach to avoid overflow/underflow
        scores = torch.log(probs + 1e-10)  # Add small epsilon to avoid log(0)
        
        if self.verbose:
            # Get top tokens after weighting
            top_probs, top_indices = torch.topk(probs[0], 5)
            print("Top 5 next tokens after weighting:")
            for i, (token_id, prob) in enumerate(zip(top_indices.tolist(), top_probs.tolist())):
                token = tokenizer.decode([token_id])
                print(f"  {i+1}. Token ID {token_id} → '{token}' (prob: {prob:.6f})")
            
            if not weights_applied:
                print("WARNING: No significant weight changes were applied. Check token IDs.")
            print()
        
        return scores

In [8]:
def generate_story_with_weighted_words(model_name, prompt, weighted_words=None, weight_factor=2.0, verbose=True):
    """
    Generate a story using the specified model and prompt, with proper probability weighting.
    
    Args:
        model_name (str): The name of the model to use for generation
        prompt (str): The prompt to generate the story from
        weighted_words (list): List of words to give higher/lower probability
        weight_factor (float): Factor to multiply the probabilities by (>1 increases probability, <1 decreases)
        verbose (bool): Whether to print detailed token mapping and probability information
    
    Returns:
        str: The generated story
    """
    # Load model and tokenizer
    model = AutoModelForCausalLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
    
    # Move model to appropriate device
    model = model.to(device)
    print(f"Model moved to device: {next(model.parameters()).device}")
    
    print(f"\nPrompt: {prompt}")
    
    # Process on selected device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    attention_mask = torch.ones_like(input_ids).to(device)
    
    # Create a custom logits processor if weighted words are provided
    if weighted_words and len(weighted_words) > 0:        
        # Create the logits processor
        logits_processor = LogitsProcessorList([
            ProbabilityWeightingLogitsProcessor(tokenizer, weighted_words, weight_factor, verbose)
        ])
        
        # Generate text with the logits processor
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=200,
            num_beams=1,
            do_sample=True,  # Enable sampling to see the effect of weighting
            temperature=0.9,
            logits_processor=logits_processor
        )
    else:
        # Generate text without logits processor
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=200,
            num_beams=1
        )
    
    story = tokenizer.decode(output[0], skip_special_tokens=True)
    
    print("\nGenerated story:")
    display(Markdown(story))
    
    return story


In [9]:
# Test the improved probability weighting implementation with verbose output
model_name = 'roneneldan/TinyStories-33M'
prompt = "Once upon a time, there was a"
words_to_boost = [" dog", " cat"]  # Words to emphasize
weight_factor = 5.0  # Increase probability by 10x

story_with_proper_weights = generate_story_with_weighted_words(
    model_name, 
    prompt, 
    weighted_words=words_to_boost, 
    weight_factor=weight_factor,
    verbose=False  # Show all token mapping and probability details
)


Model moved to device: mps:0

Prompt: Once upon a time, there was a


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Generated story:


Once upon a time, there was a little cat who lived in a big forest. The cat was very lazy and loved to sleep all day. One day, the cat's friend, a rabbit, came to visit her. 

"Hello, little cat! Why are you so lazy today?" asked the rabbit. 

"I don't like to wake up early," replied the cat. 

"Well, you should sleep more often. It will make you and your friends happy," said the rabbit. 

Suddenly, the cat woke up by a loud noise. She looked around and saw that a tree had fallen and blocked the path. "Oh no! What do we do now?" cried the cat. 

"Don't worry, little dog. We can use this stick to move the tree so we can continue our journey. We just need to be careful," said the cat. 

"Thank you, wise owl. You are

In [10]:
# Test with decreased probabilities for common words
model_name = 'roneneldan/TinyStories-33M'
prompt = "Once upon a time, there was a"
words_to_avoid = ["little", "Lily", "boy", "girl"]  # Common words to de-emphasize
weight_factor = 0.01  # Decrease probability by multiplying by 0.01

story_with_decreased_probs = generate_story_with_weighted_words(
    model_name, 
    prompt, 
    weighted_words=words_to_avoid, 
    weight_factor=weight_factor,
    verbose=True  # Show all token mapping and probability details
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Model moved to device: mps:0

Prompt: Once upon a time, there was a

Prompt token mapping:
Position 0: Token ID 1858 → 'There'
Position 1: Token ID 1752 → ' once'
Position 2: Token ID 373 → ' was'
Position 3: Token ID 257 → ' a'
Position 4: Token ID 582 → ' man'
Position 5: Token ID 508 → ' who'
Position 6: Token ID 373 → ' was'
Position 7: Token ID 257 → ' a'
Position 8: Token ID 1049 → ' great'
Position 9: Token ID 19177 → ' hunter'
Position 10: Token ID 13 → '.'
Position 11: Token ID 679 → ' He'
Position 12: Token ID 373 → ' was'
Position 13: Token ID 1900 → ' known'
Position 14: Token ID 329 → ' for'
Position 15: Token ID 465 → ' his'
Position 16: Token ID 5032 → ' skill'
Position 17: Token ID 290 → ' and'
Position 18: Token ID 220 → ' '

Weighted words token mapping:
Word 'little':
  Without space prefix: [31629] → 'little'
  With space prefix: [1310] → ' little'
Word 'Lily':
  Without space prefix: [43, 813] → 'Lily'
  With space prefix: [20037] → ' Lily'
Word 'boy':
  Without sp

Once upon a time, there was a lazy bear who lived deep in the forest. Every day, he would sunbathe in the warm sunshine and enjoy the quiet of the season.

One day, while he was feeling particularly lazy, he had an idea. He decided to take a big walk and explore the forest! As he was walking, he was almost feeling more and more tired.

Suddenly, a big storm rolled in, and the rain started to pour down. The bear was so startled that he quickly ran back home.

When he got home, he was soaking wet and his fur was all wet and muddy! When he went to sleep, his fur was soaking tight and dripping from his fur.

The next day, the bear still hadn't been able to explore the forest, but he felt even more lazy and happy. He never wanted to go on a walk again!


In [11]:
# Debug token IDs for problematic words
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")

# Check the token IDs for the word "little"
words_to_check = ["little", "big", "small"]
print("Token ID mapping for problematic words:")
for word in words_to_check:
    # Check with and without space prefix
    word_ids = tokenizer.encode(word, add_special_tokens=False)
    word_with_space_ids = tokenizer.encode(" " + word, add_special_tokens=False)
    
    print(f"Word '{word}':")
    print(f"  Without space prefix: {word_ids} → '{tokenizer.decode(word_ids)}'")
    print(f"  With space prefix: {word_with_space_ids} → '{tokenizer.decode(word_with_space_ids)}'")
    
    # Show individual tokens if multiple
    if len(word_ids) > 1:
        print(f"  Individual tokens without space:")
        for token_id in word_ids:
            print(f"    Token ID {token_id} → '{tokenizer.decode([token_id])}'")
    
    if len(word_with_space_ids) > 1:
        print(f"  Individual tokens with space:")
        for token_id in word_with_space_ids:
            print(f"    Token ID {token_id} → '{tokenizer.decode([token_id])}'")
    print()


Token ID mapping for problematic words:
Word 'little':
  Without space prefix: [31629] → 'little'
  With space prefix: [1310] → ' little'

Word 'big':
  Without space prefix: [14261] → 'big'
  With space prefix: [1263] → ' big'

Word 'small':
  Without space prefix: [17470] → 'small'
  With space prefix: [1402] → ' small'

